# N25 — Pace Agent

The Pace Agent answers one question: **how fast are we going this lap?**

It wraps the N06 XGBoost delta-lap-time model into a clean agent interface. Given the current race state (tyre compound, tyre life, lap number, track conditions, team/driver), it returns a lap time prediction with a confidence interval derived from bootstrap quantiles.

This is the first of seven sub-agents in the multi-agent strategy system (N25–N31). Its output feeds directly into the Strategy Orchestrator (N31) as the pace signal used to evaluate undercut/overcut windows and stint extension decisions.

## Pipeline position

<pre>
race_state (lap_number, tyre_life, compound, team, ...)
    │
    ▼
N25 — Pace Agent
    │
    ├── lap_time_pred        predicted lap time (seconds)
    ├── delta_vs_median      delta vs session median pace (+ = slower)
    └── confidence_interval  [p10, p90] bootstrap interval
</pre>




## Models used
- **N06** — XGBoost delta lap time (`data/models/lap_time/`)

## Steps
- **Step 0** — Setup & model loading
- **Step 1** — Feature preparation (race state → model input)
- **Step 2** — Inference + bootstrap confidence interval
- **Step 3** — `run_pace_agent(lap_state) → PaceOutput`
- **Step 4** — Demo: full 2025 race replay (lap by lap)
- **Step 5** — Export schema & config

---

In [2]:
# ── Step 0: Setup & model loading ──────────────────────────────────────────
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / '.git').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import numpy as np
import pandas as pd
import xgboost as xgb

# ── Paths ───────────────────────────────────────────────────────────────────
MODELS_DIR  = repo_root / 'data' / 'models' / 'lap_time'
OUTPUTS_DIR = repo_root / 'notebooks' / 'agents' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load N06 model + features ───────────────────────────────────────────────
def load_pace_model(models_dir=MODELS_DIR):
    features = json.loads((models_dir / 'xgb_laptime_delta_feature_names.json').read_text())
    model = xgb.XGBRegressor()
    model.load_model(models_dir / 'xgb_laptime_delta_final.json')
    return model, features

MODEL, FEATURES = load_pace_model()

print(f"Model loaded — {len(FEATURES)} features")
print(f"Features: {FEATURES}")


Model loaded — 25 features
Features: ['DriverNumber', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'Position', 'CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad', 'Year', 'FuelEffect', 'Prev_LapTime', 'Prev_TyreLife', 'Prev_SpeedST', 'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall', 'laps_remaining', 'Cluster', 'mean_sector_speed', 'Prev_DegradationRate', 'Prev_CumulativeDeg', 'Prev_DegAcceleration']


---